# [Step 4] EDA 반영 전처리 및 파생 피쳐(Feature) 생성

In [24]:
import pandas as pd
import numpy as np
from pathlib import Path
BASE_DIR = Path(".")
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
def build_multivariate_features(file_path, output_name):
    df = pd.read_csv(file_path)
    df = df.sort_values(["ticker", "date"]).copy()
    df["prev_volume"] = df.groupby("ticker")["volume"].shift(1)
    df["vol_chg_rate"] = (df["volume"] - df["prev_volume"]) / df["prev_volume"]
    df["volume_ma20_ratio"] = df["volume"] / df.groupby("ticker")["volume"].transform(lambda x: x.rolling(20).mean())
    df["prev_close"] = df.groupby("ticker")["close"].shift(1)
    df["daily_return"] = (df["close"] - df["prev_close"]) / df["prev_close"]
    df["volatility_5d"] = df.groupby("ticker")["daily_return"].transform(lambda x: x.rolling(5).std())
    df["drawdown_after_peak_5d"] = (df.groupby("ticker")["high"].transform(lambda x: x.rolling(5).max()) - df["close"]) / df.groupby("ticker")["high"].transform(lambda x: x.rolling(5).max())
    candle_range = (df["high"] - df["low"]).replace(0, np.nan)
    df["upper_shadow_ratio"] = (df["high"] - df[["open", "close"]].max(axis=1)) / candle_range
    df["body_ratio"] = (df["close"] - df["open"]).abs() / candle_range
    df["upper_shadow_streak_5d"] = df.groupby("ticker")["upper_shadow_ratio"].transform(lambda x: (x > 0.4).astype(int).rolling(5).sum())
    df.dropna(subset=["volume_ma20_ratio", "volatility_5d", "upper_shadow_streak_5d"], inplace=True)
    df.to_csv(DATA_PROCESSED / f"{output_name}_features.csv", index=False, encoding="utf-8-sig")
    return df
train_fe = build_multivariate_features(DATA_RAW / "train.csv", "train")
valid_fe = build_multivariate_features(DATA_RAW / "valid.csv", "valid")
test_fe = build_multivariate_features(DATA_RAW / "test.csv", "test")
print("[+] Step 4: 다변량 파생 피쳐셋(Train/Valid/Test) 생성 완료")